<a href="https://colab.research.google.com/github/jamesreddeviltna/Open-House/blob/main/_Gold.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gold Price Movement Prediction Tutorial

This notebook demonstrates how to:
1. Retrieve historical gold price data using the Yahoo Finance API (`yfinance`).
2. Clean, process, and engineer features from the time series data.
3. Train a simple machine learning model (Random Forest) to predict whether the gold price will go up or down.
4. Visualize the actual historical prices alongside our model's predictions.
5. Wrap the entire workflow into a function and build an interactive `ipywidgets` interface.

In [ ]:
# Install necessary libraries
!pip install yfinance ipywidgets

## Step 1: Fetching Gold Price Data
We will use `yfinance` to download historical price data for Gold (Ticker: `GC=F`).

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

# Automatically set today's date as the end date
today_str = datetime.today().strftime('%Y-%m-%d')

# Download historical gold data
gold_data = yf.download('GC=F', start='2023-01-01', end=today_str)
display(gold_data)

## Step 2: Feature Engineering & Target Labeling
We will predict whether tomorrow's closing price will be higher (1) or lower (0) than today's. We will use simple features like moving averages and daily returns.

In [ ]:
# Squeeze the data columns to ensure they are 1D Series if MultiIndex is present
close_series = gold_data['Close'].squeeze()
return_series = gold_data['Return'].squeeze() if 'Return' in gold_data else close_series.pct_change()

# Prepare target: 1 if next day's Close is higher than today's Close, else 0
gold_data['Target'] = (close_series.shift(-1) > close_series).astype(int)

# Basic Features
gold_data['Return'] = return_series
gold_data['MA_5'] = close_series.rolling(window=5).mean()
gold_data['MA_20'] = close_series.rolling(window=20).mean()
gold_data['Volatility_5'] = return_series.rolling(window=5).std()

# User requested features
gold_data['Volatility_20'] = return_series.rolling(window=20).std()
gold_data['Diff_Close_MA5'] = (close_series - gold_data['MA_5'].squeeze()) / gold_data['MA_5'].squeeze()
gold_data['Diff_Close_MA20'] = (close_series - gold_data['MA_20'].squeeze()) / gold_data['MA_20'].squeeze()
gold_data['Diff_MA5_MA20'] = (gold_data['MA_5'].squeeze() - gold_data['MA_20'].squeeze()) / gold_data['MA_20'].squeeze()
gold_data['Return_5'] = close_series.pct_change(periods=5)
gold_data['Return_20'] = close_series.pct_change(periods=20)

# Drop NaN values generated by shifting and rolling calculations
gold_data.dropna(inplace=True)
display(gold_data[['Close', 'Target', 'Return', 'MA_5', 'MA_20', 'Volatility_5', 'Volatility_20', 'Diff_Close_MA5', 'Diff_Close_MA20', 'Diff_MA5_MA20', 'Return_5', 'Return_20']].head())

## Step 3: Model Training and Prediction
We will split the data into training and test sets chronologically to respect the time series nature, then train a Random Forest Classifier.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Define features and target including the simple ones
features = [
    'Return', 'Volatility_5', 'Volatility_20',
    'Diff_Close_MA5', 'Diff_Close_MA20', 'Diff_MA5_MA20',
    'Return_5', 'Return_20'
]
X = gold_data[features]
y = gold_data['Target']

# Chronological split: 80% train, 20% test
split_idx = int(len(gold_data) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

# Train Random Forest model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Predict and evaluate
test_preds = model.predict(X_test)
accuracy = accuracy_score(y_test, test_preds)
print(f"Directional Prediction Accuracy: {accuracy:.2%}")

## Step 4: Visualizing Predictions and Historical Data
Let's visualize the actual close prices in the test set, marking days where our model predicts a price increase.

In [ ]:
test_data = gold_data.iloc[split_idx:].copy()
test_data['Prediction'] = test_preds

plt.figure(figsize=(14, 7))
plt.plot(test_data.index, test_data['Close'], label='Actual Gold Price', color='grey', alpha=0.6)

# Highlight predicted UP days in green and DOWN days in red
up_days = test_data[test_data['Prediction'] == 1]
down_days = test_data[test_data['Prediction'] == 0]

plt.scatter(up_days.index, up_days['Close'], color='green', marker='^', label='Predicted Up', alpha=1, s=40)
plt.scatter(down_days.index, down_days['Close'], color='red', marker='v', label='Predicted Down', alpha=1, s=40)

plt.title('Gold Price Close and Directional Predictions')
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.legend()
plt.grid(True)
plt.show()

## Step 5: Wrapping into a Reusable Function
We combine the components into a single function that dynamically slices training and prediction (test) ranges.

In [ ]:
def train_predict_gold(train_start, train_end, pred_start, pred_end):
    # Fetch a wider range to avoid boundary issues during indicator calculation
    fetch_start = pd.to_datetime(train_start) - pd.Timedelta(days=30)
    data = yf.download('GC=F', start=fetch_start, end=pred_end, progress=False)

    if data.empty:
        print("No data retrieved for selected date range.")
        return

    # Ensure we use 1D Series even with MultiIndex column headers
    close_series = data['Close'].squeeze()

    # Feature Engineering
    data['Target'] = (close_series.shift(-1) > close_series).astype(int)
    data['Return'] = close_series.pct_change()
    return_series = data['Return'].squeeze()

    data['MA_5'] = close_series.rolling(window=5).mean()
    data['MA_20'] = close_series.rolling(window=20).mean()
    data['Volatility_5'] = return_series.rolling(window=5).std()

    # Dynamic Features
    data['Volatility_20'] = return_series.rolling(window=20).std()
    data['Diff_Close_MA5'] = (close_series - data['MA_5'].squeeze()) / data['MA_5'].squeeze()
    data['Diff_Close_MA20'] = (close_series - data['MA_20'].squeeze()) / data['MA_20'].squeeze()
    data['Diff_MA5_MA20'] = (data['MA_5'].squeeze() - data['MA_20'].squeeze()) / data['MA_20'].squeeze()
    data['Return_5'] = close_series.pct_change(periods=5)
    data['Return_20'] = close_series.pct_change(periods=20)

    data.dropna(inplace=True)

    # Split based on specific dates
    train_mask = (data.index >= train_start) & (data.index <= train_end)
    pred_mask = (data.index >= pred_start) & (data.index <= pred_end)

    train_set = data[train_mask]
    pred_set = data[pred_mask]

    if train_set.empty or pred_set.empty:
        print("Training or prediction set is empty. Choose broader dates.")
        return

    features = [
        'Return', 'Volatility_5', 'Volatility_20',
        'Diff_Close_MA5', 'Diff_Close_MA20', 'Diff_MA5_MA20',
        'Return_5', 'Return_20'
    ]
    X_train, y_train = train_set[features], train_set['Target']
    X_pred, y_pred = pred_set[features], pred_set['Target']

    # Fit and Predict
    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    clf.fit(X_train, y_train)
    predictions = clf.predict(X_pred)

    pred_set = pred_set.copy()
    pred_set['Prediction'] = predictions

    # Plot
    plt.figure(figsize=(14, 6))
    plt.plot(pred_set.index, pred_set['Close'].squeeze(), label='Actual Price', color='grey', alpha=0.7)

    up = pred_set[pred_set['Prediction'] == 1]
    down = pred_set[pred_set['Prediction'] == 0]

    plt.scatter(up.index, up['Close'].squeeze(), color='green', marker='^', label='Predict Up', s=50)
    plt.scatter(down.index, down['Close'].squeeze(), color='red', marker='v', label='Predict Down', s=50)

    plt.title(f'Predictions from {pred_start} to {pred_end}')
    plt.xlabel('Date')
    plt.ylabel('Price (USD)')
    plt.legend()
    plt.grid(True)
    plt.show()

## Step 6: Interactive UI with `ipywidgets`
We create a dashboard using date pickers so you can easily modify the training and validation horizons.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML
from datetime import datetime, timedelta
import matplotlib.pyplot as plt

# Define simplified widgets
train_start_picker = widgets.DatePicker(description='Train Start', value=pd.to_datetime('2023-01-01'))
button = widgets.Button(description="Run Trend Prediction", button_style='success')
output = widgets.Output()

def on_button_click(b):
    with output:
        output.clear_output()

        # Define dynamic time boundaries
        today = datetime.today()
        yesterday = today - timedelta(days=1)

        train_start = train_start_picker.value.strftime('%Y-%m-%d')
        train_end = yesterday.strftime('%Y-%m-%d')

        # Fetch data
        fetch_start = pd.to_datetime(train_start) - pd.Timedelta(days=30)
        data = yf.download('GC=F', start=fetch_start, end=today.strftime('%Y-%m-%d'), progress=False)

        if data.empty:
            print("No data retrieved. Please adjust your Train Start date.")
            return

        # Feature Engineering
        close_series = data['Close'].squeeze()
        data['Target'] = (close_series.shift(-1) > close_series).astype(int)
        data['Return'] = close_series.pct_change()
        return_series = data['Return'].squeeze()

        data['MA_5'] = close_series.rolling(window=5).mean()
        data['MA_20'] = close_series.rolling(window=20).mean()
        data['Volatility_5'] = return_series.rolling(window=5).std()
        data['Volatility_20'] = return_series.rolling(window=20).std()
        data['Diff_Close_MA5'] = (close_series - data['MA_5'].squeeze()) / data['MA_5'].squeeze()
        data['Diff_Close_MA20'] = (close_series - data['MA_20'].squeeze()) / data['MA_20'].squeeze()
        data['Diff_MA5_MA20'] = (data['MA_5'].squeeze() - data['MA_20'].squeeze()) / data['MA_20'].squeeze()
        data['Return_5'] = close_series.pct_change(periods=5)
        data['Return_20'] = close_series.pct_change(periods=20)

        data.dropna(inplace=True)

        if data.empty:
            print("Not enough data to construct features. Choose an earlier Train Start date.")
            return

        # Split
        train_set = data[data.index <= train_end]

        if train_set.empty:
            print("Training set is empty. Choose a valid Train Start date.")
            return

        features = [
            'Return', 'Volatility_5', 'Volatility_20',
            'Diff_Close_MA5', 'Diff_Close_MA20', 'Diff_MA5_MA20',
            'Return_5', 'Return_20'
        ]

        X_train, y_train = train_set[features], train_set['Target']

        # Train Classifier
        clf = RandomForestClassifier(n_estimators=100, random_state=42)
        clf.fit(X_train, y_train)

        # Predict for tomorrow using the most recent available data point
        latest_data = data.tail(1)
        latest_date = latest_data.index[0].strftime('%Y-%m-%d')
        X_latest = latest_data[features]

        prediction = clf.predict(X_latest)[0]

        # Display the prediction result
        if prediction == 1:
            display(HTML(f"<h3>🔮 Gold Trend Prediction for Tomorrow: <span style='color:green;'>▲ UP (Bullish)</span></h3>"))
        else:
            display(HTML(f"<h3>🔮 Gold Trend Prediction for Tomorrow: <span style='color:red;'>▼ DOWN (Bearish)</span></h3>"))

        print(f"Model trained from {train_start} to {train_end} using the latest metrics from {latest_date}.")

        # Create subplots to visualize past history: Prices + MAs and Volatilities
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

        # Plot Price and MAs
        ax1.plot(data.index, close_series.loc[data.index], label='Actual Close Price', color='grey', alpha=0.8, linewidth=1.5)
        ax1.plot(data.index, data['MA_5'], label='5-Day Moving Average', color='blue', alpha=0.7, linestyle='--')
        ax1.plot(data.index, data['MA_20'], label='20-Day Moving Average', color='orange', alpha=0.7, linestyle='-.')
        ax1.set_title('Gold Price Trend & Moving Averages')
        ax1.set_ylabel('Price (USD)')
        ax1.legend()
        ax1.grid(True, linestyle=':', alpha=0.6)

        # Plot Volatilities
        ax2.plot(data.index, data['Volatility_5'], label='5-Day Rolling Volatility', color='purple', alpha=0.7)
        ax2.plot(data.index, data['Volatility_20'], label='20-Day Rolling Volatility', color='teal', alpha=0.7)
        ax2.set_title('Gold Rolling Volatility Metrics')
        ax2.set_xlabel('Date')
        ax2.set_ylabel('Volatility (Std Dev)')
        ax2.legend()
        ax2.grid(True, linestyle=':', alpha=0.6)

        plt.tight_layout()
        plt.show()

button.on_click(on_button_click)

# Layout UI
ui = widgets.VBox([
    widgets.HBox([train_start_picker, button]),
    output
])
display(ui)

prompts (exclude debugging):


- A simple program (could be in many cells for step-by-step running) (+markdowns) for learning to use API to retrieve the available gold price data. Then, use some simple methods to learn from the timeseries data and predict future prices (whether the price will go up or down). Visualize the data and prediction result together in a single graph After the step-by-step learning, wrap the program as a function of training and prediction time period. Another cell is used to create ipywidgets to specify training and prediction time period before running the prediction and visualization.

- add the features of the Volatility_20 along with the relative difference 1) between 'Close' and 'MA_5' 2) between 'Close' and 'MA_20' 3) between 'MA_5' and 'MA_20'

- also add return_5 and return_20

- automatically specify the end date as today

- input only the train start and also print out (very clear message) the trend for tomorrow

- also visualize the graph of past data and display the MA and volatility


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from datetime import datetime

# Dynamic widgets for stochastic simulation
train_start_picker_gbm = widgets.DatePicker(description='Train Start', value=pd.to_datetime('2023-01-01'))
days_slider = widgets.IntSlider(value=7, min=5, max=10, step=1, description='Days Ahead:')
sim_button = widgets.Button(description="Run Stochastic GBM", button_style='info')
stoch_output = widgets.Output()

def run_gbm_simulation(b):
    with stoch_output:
        stoch_output.clear_output()

        # Define dynamic time boundaries
        today = datetime.today()
        train_start = train_start_picker_gbm.value.strftime('%Y-%m-%d')
        today_str = today.strftime('%Y-%m-%d')

        # Fetch data up to today
        data = yf.download('GC=F', start=train_start, end=today_str, progress=False)

        if data.empty:
            print("No data retrieved. Please adjust your Train Start date.")
            return

        close_prices = data['Close'].squeeze()
        returns = close_prices.pct_change().dropna()

        if len(returns) < 5:
            print("Too few data points to estimate model parameters. Choose an earlier Train Start date.")
            return

        # Calculate parameters for Geometric Brownian Motion
        mu = returns.mean()          # Daily Drift
        sigma = returns.std()        # Daily Volatility
        S0 = close_prices.iloc[-1]   # Current Price

        N_days = days_slider.value
        N_sims = 1000                # Number of simulation paths
        dt = 1                       # Time step dt = 1 day

        # Generate simulated paths
        sim_paths = np.zeros((N_days + 1, N_sims))
        sim_paths[0] = S0

        for t in range(1, N_days + 1):
            Z = np.random.standard_normal(N_sims)
            sim_paths[t] = sim_paths[t-1] * np.exp((mu - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z)

        # Dates for prediction
        last_date = close_prices.index[-1]
        pred_dates = [last_date + pd.Timedelta(days=i) for i in range(N_days + 1)]

        # Plot the stochastic projection
        plt.figure(figsize=(12, 6))

        # Plot recent actual data (last 30 days)
        recent_prices = close_prices.tail(30)
        plt.plot(recent_prices.index, recent_prices.values, color='black', label='Actual Close Price', linewidth=2)

        # Plot simulated paths
        for i in range(min(50, N_sims)):
            plt.plot(pred_dates, sim_paths[:, i], color='cyan', alpha=0.15, linewidth=1)

        # Plot mean path
        mean_path = np.mean(sim_paths, axis=1)
        plt.plot(pred_dates, mean_path, color='red', label='Expected Path (Mean)', linewidth=2.5)

        # Plot 95% Confidence Intervals
        lower_bound = np.percentile(sim_paths, 2.5, axis=1)
        upper_bound = np.percentile(sim_paths, 97.5, axis=1)
        plt.plot(pred_dates, lower_bound, color='red', linestyle=':', label='95% Confidence Interval')
        plt.plot(pred_dates, upper_bound, color='red', linestyle=':')
        plt.fill_between(pred_dates, lower_bound, upper_bound, color='red', alpha=0.1)

        plt.title(f'Geometric Brownian Motion (GBM) Stochastic Projection for {N_days} Days')
        plt.xlabel('Date')
        plt.ylabel('Gold Price (USD)')
        plt.legend(loc='upper left')
        plt.grid(True, linestyle=':', alpha=0.6)
        plt.show()

        # Summary Statistics
        print(f"--- GBM Simulation Summary ({N_days}-Day Forecast) ---")
        print(f"Data Period: {train_start} to {last_date.strftime('%Y-%m-%d')}")
        print(f"Current Gold Price (S0): ${S0:.2f}")
        print(f"Estimated Daily Drift (mu): {mu:.5%}")
        print(f"Estimated Daily Volatility (sigma): {sigma:.5%}")
        print(f"Expected Price on {pred_dates[-1].strftime('%Y-%m-%d')}: ${mean_path[-1]:.2f}")
        print(f"95% Price Range: ${lower_bound[-1]:.2f} to ${upper_bound[-1]:.2f}")

sim_button.on_click(run_gbm_simulation)

# Render Dashboard
display(widgets.VBox([
    widgets.HBox([train_start_picker_gbm, days_slider, sim_button]),
    stoch_output
]))

prompt: with the same gold data. Now try a regression model employing some simple stochastic calculus techniques and prodict the price ahead for 5-10 days (adjustable).